# 01 — Instrument calibration (prereg A8 §c + §e)

## 1. Verify the GPU(s)

In [ ]:
!nvidia-smi

## 2. Clone the repo

Set `PROBE_REPO_URL` to your public Git URL or `PROBE_REPO_DIR` to a local checkout.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_DIR = os.environ.get("PROBE_REPO_DIR", "/kaggle/working/probe-capacity-invariance")
if not (Path(REPO_DIR) / "pyproject.toml").is_file():
    repo_url = os.environ.get("PROBE_REPO_URL")
    if not repo_url:
        raise RuntimeError("Set PROBE_REPO_URL or PROBE_REPO_DIR before running this notebook.")
    subprocess.run(["git", "clone", repo_url, REPO_DIR], check=True)
os.chdir(REPO_DIR)


## 3. Install dependencies

In [ ]:
!pip install -q -e . --no-deps
!pip install -q h5py

In [ ]:
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "| device count:", torch.cuda.device_count())

## 4. Download shapes3d + dsprites + build the image cache

In [ ]:
!cd /kaggle/working/probe-capacity-invariance && python -m src.data.shapes3d --download --build-cache
!cd /kaggle/working/probe-capacity-invariance && python -m src.data.dsprites --download --build-cache

In [ ]:


import shutil
from pathlib import Path

REPO = Path("/kaggle/working/probe-capacity-invariance"); INPUT = Path("/kaggle/input")
restored = 0
for src in list(INPUT.glob("*/results")) + list(INPUT.glob("*/probe-capacity-invariance/results")):
    for f in src.rglob("*"):
        if f.is_file() and f.suffix in (".npz", ".json", ".jsonl"):
            dst = REPO / "results" / f.relative_to(src)
            if not dst.exists():
                dst.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(f, dst); restored += 1
print(f"restored {restored} prior result files")

## 7. Shapes3D calibration

In [ ]:
!cd /kaggle/working/probe-capacity-invariance && python -m src.probes.instrument_calibration \
    --dataset shapes3d --probe-train-sizes 2000 5000 10000 40000 \
    --random-seed 0 1 2 --device cuda --num-workers 2 --out-root results/calibration

## 8. dSprites calibration (the position arm's dataset)

In [ ]:
!cd /kaggle/working/probe-capacity-invariance && python -m src.probes.instrument_calibration \
    --dataset dsprites --probe-train-sizes 2000 5000 10000 40000 \
    --random-seed 0 1 2 --device cuda --num-workers 2 --out-root results/calibration

## 9. Read the verdict

In [ ]:
import json
from pathlib import Path
for ds in ("shapes3d", "dsprites"):
    p = Path("/kaggle/working/probe-capacity-invariance") / f"results/calibration/calibration_{ds}.json"
    if not p.exists():
        print(f"{ds}: not run"); continue
    r = json.loads(p.read_text())
    print(f"\n=== {ds} ===\nVERDICT: {r['verdict']}")
    hdr = f"{'regime':16s}{'n':>7s}{'role':>9s}{'top floor':>11s}{'headroom':>10s}{'lin gap':>9s}"
    print(hdr)
    for row in r["results"]:
        print(f"{row['regime']:16s}{row['probe_train_size']:>7d}{row['encoder_role']:>9s}"
              f"{row['top_rung_floor']:>11.4f}{row['headroom_at_top']:>10.4f}"
              f"{row['linear_rung_gap']:>+9.4f}"
              + ("  <- SATURATED" if row["saturated_at_top"] else ""))

In [ ]:



import shutil
from pathlib import Path
src = Path("/kaggle/working/probe-capacity-invariance/results"); dst = Path("/kaggle/working/results")
shutil.rmtree(dst, ignore_errors=True); shutil.copytree(src, dst)
print(f"persisted {sum(1 for _ in dst.rglob('*') if _.is_file())} files "
      f"-> click 'Save Version' now")